<a href="https://colab.research.google.com/github/megamiro-code/battlefield/blob/main/%E5%85%B5%E5%A3%AB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# CELL 1
# JAX RTS ENVIRONMENT
# 32 battlefields in parallel
# ============================================================

import jax
import jax.numpy as jnp
from jax import random
import time

print("JAX version :", jax.__version__)
print("Backend     :", jax.default_backend())
print("Devices     :", jax.devices())

# ============================================================
# CONFIG
# ============================================================

N_ENVS = 32
FIELD_SIZE = 10.0
HALF_FIELD = FIELD_SIZE / 2.0
N_SOLDIERS_PER_TEAM = 100
N_SOLDIERS_TOTAL = N_SOLDIERS_PER_TEAM * 2
N_COMMANDERS = 2
N_UNITS = N_SOLDIERS_TOTAL + N_COMMANDERS
DT = 0.20
MAX_TIME = 180.0
MAX_STEPS = int(MAX_TIME / DT)
ATTACK_RANGE = 0.42
ATTACK_COOLDOWN = 0.55
ATTACK_DAMAGE = 0.20
SOLDIER_RADIUS = 0.13
COMMANDER_RADIUS = 0.22
COMMANDER_EXTRA_MARGIN = 0.10
SOLDIER_SPEED_MIN = 0.40
SOLDIER_SPEED_MAX = 0.65
N_FEATURES_PER_UNIT = 10
TERRAIN_RES = 10

OBS_SIZE = TERRAIN_RES * TERRAIN_RES + N_UNITS * N_FEATURES_PER_UNIT
ACTION_SIZE = N_SOLDIERS_PER_TEAM * 3

# 密な報酬シェーピング
# 与ダメージ - 被ダメージを小さく毎ステップ加算
SHAPING_COEF = 0.005

print()
print("Observation size :", OBS_SIZE)
print("Action size      :", ACTION_SIZE)
print("Simulation steps :", MAX_STEPS)
print("Parallel battles :", N_ENVS)
print("Total soldiers   :", N_SOLDIERS_TOTAL)
print("Total units      :", N_UNITS)
print("Shaping coef     :", SHAPING_COEF)

# ============================================================
# TERRAIN
# ============================================================

walls = jnp.array([
    [-4.5, -4.5], [-4.5, 4.5], [4.5, -4.5], [4.5, 4.5],
    [-0.5, -0.5], [-0.5, 0.5], [0.5, -0.5], [0.5, 0.5],
], dtype=jnp.float32)

def make_terrain():
    xs = jnp.arange(TERRAIN_RES) - 4.5
    zs = jnp.arange(TERRAIN_RES) - 4.5
    xx, zz = jnp.meshgrid(xs, zs)
    centers = jnp.stack([xx.reshape(-1), zz.reshape(-1)], axis=-1)

    def blocked(cell_index):
        cell = centers[cell_index]
        d = jnp.abs(cell[None, :] - walls)
        hit = jnp.all(d < 0.5, axis=1)
        return jnp.any(hit)

    terrain = jax.vmap(blocked)(jnp.arange(TERRAIN_RES * TERRAIN_RES))
    return terrain.astype(jnp.float32)

terrain = make_terrain()
print("Terrain shape:", terrain.shape)

# ============================================================
# UNIT INDEX LAYOUT
# ============================================================

RED_COMMANDER_INDEX = 0
BLUE_COMMANDER_INDEX = 1
RED_SOLDIER_START = 2
RED_SOLDIER_END = 102
BLUE_SOLDIER_START = 102
BLUE_SOLDIER_END = 202

RED_SOLDIER_INDICES = jnp.arange(RED_SOLDIER_START, RED_SOLDIER_END)
BLUE_SOLDIER_INDICES = jnp.arange(BLUE_SOLDIER_START, BLUE_SOLDIER_END)
ALL_SOLDIER_INDICES = jnp.arange(RED_SOLDIER_START, BLUE_SOLDIER_END)
ALL_UNIT_INDICES = jnp.arange(N_UNITS)

# ============================================================
# CONSTANT UNIT ATTRIBUTES
# ============================================================

teams = jnp.concatenate([
    jnp.array([0.0, 1.0]),
    jnp.zeros(N_SOLDIERS_PER_TEAM),
    jnp.ones(N_SOLDIERS_PER_TEAM)
])

soldier_mask = jnp.concatenate([
    jnp.zeros(N_COMMANDERS),
    jnp.ones(N_SOLDIERS_TOTAL)
])

commander_mask = 1.0 - soldier_mask

# ============================================================
# RESET
# ============================================================

def reset_one(key):
    k1, k2, k3, k4, k5 = random.split(key, 5)

    x = jnp.zeros(N_UNITS, dtype=jnp.float32)
    z = jnp.zeros(N_UNITS, dtype=jnp.float32)
    vx = jnp.zeros(N_UNITS, dtype=jnp.float32)
    vz = jnp.zeros(N_UNITS, dtype=jnp.float32)
    hp = jnp.ones(N_UNITS, dtype=jnp.float32)
    alive = jnp.ones(N_UNITS, dtype=jnp.float32)
    attack_timer = jnp.zeros(N_UNITS, dtype=jnp.float32)
    speed = jnp.zeros(N_UNITS, dtype=jnp.float32)

    x = x.at[RED_COMMANDER_INDEX].set(-3.2)
    z = z.at[RED_COMMANDER_INDEX].set(0.0)
    x = x.at[BLUE_COMMANDER_INDEX].set(3.2)
    z = z.at[BLUE_COMMANDER_INDEX].set(0.0)

    speed = speed.at[RED_COMMANDER_INDEX].set(0.20)
    speed = speed.at[BLUE_COMMANDER_INDEX].set(0.20)

    red_x = random.uniform(
        k1, (N_SOLDIERS_PER_TEAM,),
        minval=-4.3, maxval=-0.8
    )
    red_z = random.uniform(
        k2, (N_SOLDIERS_PER_TEAM,),
        minval=-4.3, maxval=4.3
    )
    blue_x = random.uniform(
        k3, (N_SOLDIERS_PER_TEAM,),
        minval=0.8, maxval=4.3
    )
    blue_z = random.uniform(
        k4, (N_SOLDIERS_PER_TEAM,),
        minval=-4.3, maxval=4.3
    )
    soldier_speeds = random.uniform(
        k5, (N_SOLDIERS_TOTAL,),
        minval=SOLDIER_SPEED_MIN,
        maxval=SOLDIER_SPEED_MAX
    )

    x = x.at[RED_SOLDIER_START:RED_SOLDIER_END].set(red_x)
    z = z.at[RED_SOLDIER_START:RED_SOLDIER_END].set(red_z)
    x = x.at[BLUE_SOLDIER_START:BLUE_SOLDIER_END].set(blue_x)
    z = z.at[BLUE_SOLDIER_START:BLUE_SOLDIER_END].set(blue_z)
    speed = speed.at[RED_SOLDIER_START:BLUE_SOLDIER_END].set(soldier_speeds)

    return {
        "x": x, "z": z, "vx": vx, "vz": vz,
        "hp": hp, "alive": alive,
        "attack_timer": attack_timer, "speed": speed,
        "time": jnp.array(0.0, dtype=jnp.float32),
        "done": jnp.array(False),
    }

reset_parallel = jax.jit(jax.vmap(reset_one))

def reset(key):
    keys = random.split(key, N_ENVS)
    return reset_parallel(keys)

# ============================================================
# OBSERVATION
# ============================================================

def observation_one(state, perspective_team):
    x, z = state["x"], state["z"]
    vx, vz = state["vx"], state["vz"]
    hp, alive = state["hp"], state["alive"]

    own = (teams == perspective_team)
    enemy = ~own

    unit_features = jnp.stack([
        x / HALF_FIELD, z / HALF_FIELD,
        vx, vz, hp,
        own.astype(jnp.float32),
        enemy.astype(jnp.float32),
        soldier_mask,
        commander_mask,
        alive,
    ], axis=-1)

    unit_features = unit_features.reshape(-1)
    return jnp.concatenate([terrain, unit_features])

observation = jax.jit(observation_one)

# ============================================================
# ACTION DECODING
# ============================================================

def decode_actions(action):
    action = action.reshape(N_SOLDIERS_PER_TEAM, 3)
    raw_dx = action[:, 0]
    raw_dz = action[:, 1]
    raw_attack = action[:, 2]

    norm = jnp.sqrt(raw_dx * raw_dx + raw_dz * raw_dz + 1e-8)
    dx = raw_dx / norm
    dz = raw_dz / norm
    attack = (raw_attack > 0.5).astype(jnp.float32)

    return dx, dz, attack

# ============================================================
# UNIT-UNIT COLLISION
# ============================================================

def pairwise_separation(x, z, alive):
    dx = x[:, None] - x[None, :]
    dz = z[:, None] - z[None, :]
    dist2 = dx * dx + dz * dz
    dist = jnp.sqrt(dist2 + 1e-8)

    radius = (
        commander_mask * COMMANDER_RADIUS
        + soldier_mask * SOLDIER_RADIUS
    )

    required = radius[:, None] + radius[None, :]

    commander_pair = (
        (commander_mask[:, None] > 0)
        | (commander_mask[None, :] > 0)
    )

    required = required + (
        commander_pair.astype(jnp.float32)
        * COMMANDER_EXTRA_MARGIN
    )

    overlap = jnp.maximum(required - dist, 0.0)

    valid = (
        (alive[:, None] > 0)
        & (alive[None, :] > 0)
        & (dist2 > 1e-10)
    )

    correction = (
        overlap
        * valid.astype(jnp.float32)
        / (dist + 1e-8)
    )

    push_x = jnp.sum(correction * dx, axis=1)
    push_z = jnp.sum(correction * dz, axis=1)

    push_x = jnp.where(
        commander_mask > 0,
        0.0,
        push_x * 0.5
    )
    push_z = jnp.where(
        commander_mask > 0,
        0.0,
        push_z * 0.5
    )

    return push_x, push_z

# ============================================================
# UNIT-WALL COLLISION
# ============================================================

def wall_blocked(x, z, radius):
    x2 = x[:, None]
    z2 = z[:, None]
    r2 = radius[:, None]

    wx = walls[:, 0][None, :]
    wz = walls[:, 1][None, :]

    closest_x = jnp.clip(x2, wx - 0.5, wx + 0.5)
    closest_z = jnp.clip(z2, wz - 0.5, wz + 0.5)

    dx = x2 - closest_x
    dz = z2 - closest_z

    dist2 = dx * dx + dz * dz
    collision = dist2 < r2 * r2

    return jnp.any(collision, axis=1)

# ============================================================
# SINGLE ENVIRONMENT STEP
# ============================================================

def step_one(state, red_action, blue_action):
    x, z = state["x"], state["z"]
    vx, vz = state["vx"], state["vz"]
    hp, alive = state["hp"], state["alive"]
    attack_timer, speed = state["attack_timer"], state["speed"]

    red_dx, red_dz, red_attack = decode_actions(red_action)
    blue_dx, blue_dz, blue_attack = decode_actions(blue_action)

    soldier_dx = jnp.concatenate([red_dx, blue_dx])
    soldier_dz = jnp.concatenate([red_dz, blue_dz])
    soldier_attack = jnp.concatenate([red_attack, blue_attack])

    move_dx = jnp.zeros(N_UNITS, dtype=jnp.float32)
    move_dz = jnp.zeros(N_UNITS, dtype=jnp.float32)
    move_attack = jnp.zeros(N_UNITS, dtype=jnp.float32)

    move_dx = move_dx.at[ALL_SOLDIER_INDICES].set(soldier_dx)
    move_dz = move_dz.at[ALL_SOLDIER_INDICES].set(soldier_dz)
    move_attack = move_attack.at[ALL_SOLDIER_INDICES].set(soldier_attack)

    attack_timer = jnp.maximum(0.0, attack_timer - DT)
    can_move = attack_timer <= 0

    distance = speed * DT
    nx = x + move_dx * distance * can_move
    nz = z + move_dz * distance * can_move

    radius = (
        commander_mask * COMMANDER_RADIUS
        + soldier_mask * SOLDIER_RADIUS
    )

    inside = (
        (nx >= -HALF_FIELD + radius)
        & (nx <= HALF_FIELD - radius)
        & (nz >= -HALF_FIELD + radius)
        & (nz <= HALF_FIELD - radius)
    )

    blocked_wall = wall_blocked(nx, nz, radius)

    valid_move = (
        inside
        & (~blocked_wall)
        & (alive > 0)
        & can_move
    )

    nx = jnp.where(valid_move, nx, x)
    nz = jnp.where(valid_move, nz, z)
    vx = jnp.where(valid_move, move_dx, 0.0)
    vz = jnp.where(valid_move, move_dz, 0.0)

    push_x, push_z = pairwise_separation(nx, nz, alive)

    nx = nx + push_x
    nz = nz + push_z

    nx = jnp.clip(
        nx,
        -HALF_FIELD + radius,
        HALF_FIELD - radius
    )
    nz = jnp.clip(
        nz,
        -HALF_FIELD + radius,
        HALF_FIELD - radius
    )

    # ------------------------------------------------------
    # COMBAT
    # ------------------------------------------------------

    attacker_indices = ALL_SOLDIER_INDICES
    attacker_x = nx[attacker_indices]
    attacker_z = nz[attacker_indices]

    ddx = nx[None, :] - attacker_x[:, None]
    ddz = nz[None, :] - attacker_z[:, None]
    dist2 = ddx * ddx + ddz * ddz

    attacker_team = teams[attacker_indices]
    enemy_mask = teams[None, :] != attacker_team[:, None]
    target_alive = alive[None, :] > 0

    valid_target = (
        enemy_mask
        & target_alive
        & (dist2 <= ATTACK_RANGE ** 2)
    )

    same_unit = (
        ALL_UNIT_INDICES[None, :]
        == attacker_indices[:, None]
    )
    valid_target = valid_target & (~same_unit)

    target_distance = jnp.where(
        valid_target,
        dist2,
        1e9
    )

    target = jnp.argmin(target_distance, axis=1)
    has_target = jnp.any(valid_target, axis=1)

    valid_attack_command = (
        move_attack[ALL_SOLDIER_INDICES] > 0.5
    )

    can_attack = (
        valid_attack_command
        & has_target
        & (attack_timer[ALL_SOLDIER_INDICES] <= 0)
        & (alive[ALL_SOLDIER_INDICES] > 0)
    )

    damage_values = (
        ATTACK_DAMAGE
        * can_attack.astype(jnp.float32)
    )

    attack_damage = jnp.zeros(
        N_UNITS,
        dtype=jnp.float32
    )

    attack_damage = attack_damage.at[target].add(
        damage_values
    )

    # ------------------------------------------------------
    # REWARD SHAPING
    # ------------------------------------------------------

    damage_to_red = (
        jnp.sum(
            attack_damage[
                RED_SOLDIER_START:RED_SOLDIER_END
            ]
        )
        + attack_damage[RED_COMMANDER_INDEX]
    )

    damage_to_blue = (
        jnp.sum(
            attack_damage[
                BLUE_SOLDIER_START:BLUE_SOLDIER_END
            ]
        )
        + attack_damage[BLUE_COMMANDER_INDEX]
    )

    shaping_red = (
        (damage_to_blue - damage_to_red)
        / N_SOLDIERS_PER_TEAM
        * SHAPING_COEF
    )

    # ------------------------------------------------------
    # HP UPDATE
    # ------------------------------------------------------

    hp = hp - attack_damage
    dead = hp <= 0
    alive = jnp.where(dead, 0.0, alive)

    old_attack_timer = (
        attack_timer[ALL_SOLDIER_INDICES]
    )

    new_attack_timer = jnp.where(
        can_attack,
        ATTACK_COOLDOWN,
        old_attack_timer
    )

    attack_timer = attack_timer.at[
        ALL_SOLDIER_INDICES
    ].set(new_attack_timer)

    new_time = state["time"] + DT

    # ------------------------------------------------------
    # TERMINATION
    # ------------------------------------------------------

    red_commander_alive = (
        alive[RED_COMMANDER_INDEX] > 0
    )
    blue_commander_alive = (
        alive[BLUE_COMMANDER_INDEX] > 0
    )

    commander_done = (
        (~red_commander_alive)
        | (~blue_commander_alive)
    )

    timeout = new_time >= MAX_TIME
    done = commander_done | timeout

    red_soldiers_alive = jnp.sum(
        alive[
            RED_SOLDIER_START:RED_SOLDIER_END
        ]
    )

    blue_soldiers_alive = jnp.sum(
        alive[
            BLUE_SOLDIER_START:BLUE_SOLDIER_END
        ]
    )

    commander_reward = (
        red_commander_alive.astype(jnp.float32)
        - blue_commander_alive.astype(jnp.float32)
    )

    timeout_reward = (
        red_soldiers_alive - blue_soldiers_alive
    ) / N_SOLDIERS_PER_TEAM

    terminal_component_red = jnp.where(
        commander_done,
        commander_reward,
        jnp.where(
            timeout,
            timeout_reward,
            0.0
        )
    )

    reward_red = terminal_component_red + shaping_red
    reward_blue = -reward_red

    next_state = {
        "x": nx, "z": nz, "vx": vx, "vz": vz,
        "hp": hp, "alive": alive,
        "attack_timer": attack_timer,
        "speed": speed,
        "time": new_time,
        "done": done,
    }

    return (
        next_state,
        reward_red,
        reward_blue,
        done
    )

step_parallel = jax.jit(
    jax.vmap(
        step_one,
        in_axes=(0, 0, 0)
    )
)

# ============================================================
# BASIC TEST
# ============================================================

key = random.key(42)
state = reset(key)

print()
print("Initial state created.")
print("Initial times:", state["time"][:8])
assert float(jnp.max(state["time"])) == 0.0

key, red_key, blue_key = random.split(key, 3)

red_actions = random.uniform(
    red_key,
    (N_ENVS, ACTION_SIZE),
    minval=-1.0,
    maxval=1.0
)

blue_actions = random.uniform(
    blue_key,
    (N_ENVS, ACTION_SIZE),
    minval=-1.0,
    maxval=1.0
)

start = time.time()

state, rr, br, done = step_parallel(
    state,
    red_actions,
    blue_actions
)

state["time"].block_until_ready()
elapsed_first = time.time() - start

print()
print(
    "First compiled step:",
    elapsed_first,
    "sec"
)

start = time.time()

for i in range(100):
    key, red_key, blue_key = random.split(key, 3)

    red_actions = random.uniform(
        red_key,
        (N_ENVS, ACTION_SIZE),
        minval=-1.0,
        maxval=1.0
    )

    blue_actions = random.uniform(
        blue_key,
        (N_ENVS, ACTION_SIZE),
        minval=-1.0,
        maxval=1.0
    )

    state, rr, br, done = step_parallel(
        state,
        red_actions,
        blue_actions
    )

state["time"].block_until_ready()
elapsed = time.time() - start

print()
print("100 steps:", elapsed, "sec")
print("Battles:", N_ENVS)
print("Simulated seconds:", 100 * DT)
print("Red rewards:", rr[:8])
print("Blue rewards:", br[:8])
print("Done:", done[:8])
print("Final times:", state["time"][:8])
print()
print("Stage 1 test finished.")

/usr/local/lib/python3.13/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


JAX version : 0.7.2
Backend     : tpu
Devices     : [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]

Observation size : 2120
Action size      : 300
Simulation steps : 900
Parallel battles : 32
Total soldiers   : 200
Total units      : 202
Shaping coef     : 0.005
Terrain shape: (100,)

Initial state created.
Initial times: [0. 0. 0. 0. 0. 0. 0. 0.]

First compiled step: 0.9027929306030273 sec

100 steps: 0.07851815223693848 sec
Battles: 32
Simulated seconds: 20.0
Red rewards: [0.e+00 0.e+00 0.e+00 0.e+00 0.e+00 0.e+00 1.e-05 0.e+00]
Blue rewards: [-0.e+00 -0.e+00 -0.e+00 -0.e+00 -0.e+00 -0.e+00 -1.e-05 -0.e+00]
Done: [False False False False False False False False]
Final times: [20.200005 20.200005 20.200005 20.200005 20.200005 20.200005 20.200005
 20.200005]

Stage 1 test finished.


In [2]:
# ============================================================
# CELL 2
# PPO SELF-PLAY
# 対称化 + 正確な終端理由 + 密な報酬 + フルマッチ・ロールアウト
# ============================================================

import jax
import jax.numpy as jnp
from jax import random, lax
import time

# ============================================================
# PPO CONFIG
# ============================================================

HIDDEN1 = 256
HIDDEN2 = 256

GAMMA = 0.999
GAE_LAMBDA = 0.97
PPO_CLIP = 0.20
VALUE_COEF = 0.50

ENTROPY_COEF_START = 0.02
ENTROPY_COEF_END = 0.002
ENTROPY_ANNEAL_UPDATES = 200

LEARNING_RATE = 3e-4

# 900 steps = 180 sec
ROLLOUT_STEPS = MAX_STEPS

PPO_EPOCHS = 4
MINIBATCHES = 8

BATCH_SIZE = ROLLOUT_STEPS * N_ENVS * 2
MINIBATCH_SIZE = BATCH_SIZE // MINIBATCHES

def get_entropy_coef(update):
    frac = min(
        1.0,
        update / ENTROPY_ANNEAL_UPDATES
    )
    return (
        ENTROPY_COEF_START
        + (
            ENTROPY_COEF_END
            - ENTROPY_COEF_START
        ) * frac
    )

print()
print("============================================")
print("PPO SELF-PLAY")
print("============================================")
print("Observation :", OBS_SIZE)
print("Action      :", ACTION_SIZE)
print("Environments:", N_ENVS)
print("Rollout     :", ROLLOUT_STEPS)
print("PPO batch   :", BATCH_SIZE)
print("Minibatch   :", MINIBATCH_SIZE)
print("Gamma       :", GAMMA)
print("GAE lambda  :", GAE_LAMBDA)
print(
    "Rollout time:",
    ROLLOUT_STEPS * DT,
    "sec"
)
print("Max battle :", MAX_TIME, "sec")
print("Shaping coef:", SHAPING_COEF)
print(
    "Entropy coef:",
    ENTROPY_COEF_START,
    "->",
    ENTROPY_COEF_END
)
print()

# ============================================================
# REWARD VERIFICATION
# ============================================================

def expected_red_reward(
    red_commander_alive,
    blue_commander_alive,
    timeout,
    red_soldiers_alive,
    blue_soldiers_alive
):
    if red_commander_alive and not blue_commander_alive:
        return 1.0

    if not red_commander_alive and blue_commander_alive:
        return -1.0

    if timeout:
        return (
            red_soldiers_alive
            - blue_soldiers_alive
        ) / 100.0

    return 0.0

def verify_reward_logic():
    print("============================================")
    print("REWARD VERIFICATION")
    print("============================================")

    r1 = expected_red_reward(
        True, False, False, 80, 30
    )
    print("Commander kill (Red):", r1)
    assert r1 == 1.0

    r2 = expected_red_reward(
        False, True, False, 80, 30
    )
    print("Commander kill (Blue):", r2)
    assert r2 == -1.0

    r3 = expected_red_reward(
        True, True, True, 72, 54
    )
    print("Timeout 72 vs 54:", r3)
    assert abs(r3 - 0.18) < 1e-6

    r4 = expected_red_reward(
        True, True, True, 54, 72
    )
    print("Timeout 54 vs 72:", r4)
    assert abs(r4 + 0.18) < 1e-6

    r5 = expected_red_reward(
        True, True, True, 50, 50
    )
    print("Timeout 50 vs 50:", r5)
    assert r5 == 0.0

    r6 = expected_red_reward(
        True, True, False, 70, 60
    )
    print("Ongoing battle:", r6)
    assert r6 == 0.0

    print()
    print("ALL REWARD TESTS PASSED.")
    print()

verify_reward_logic()

# ============================================================
# SYMMETRIC OBSERVATION
# ============================================================

def symmetric_observation_one(
    state,
    perspective_team
):
    x, z = state["x"], state["z"]
    vx, vz = state["vx"], state["vz"]
    hp, alive = state["hp"], state["alive"]

    blue_perspective = (
        perspective_team == 1.0
    )

    transformed_x = jnp.where(
        blue_perspective,
        -x,
        x
    )

    transformed_vx = jnp.where(
        blue_perspective,
        -vx,
        vx
    )

    own = teams == perspective_team
    enemy = ~own

    unit_features = jnp.stack([
        transformed_x / HALF_FIELD,
        z / HALF_FIELD,
        transformed_vx,
        vz,
        hp,
        own.astype(jnp.float32),
        enemy.astype(jnp.float32),
        soldier_mask,
        commander_mask,
        alive,
    ], axis=-1)

    unit_features = unit_features.reshape(-1)

    terrain_2d = terrain.reshape(
        TERRAIN_RES,
        TERRAIN_RES
    )

    mirrored_terrain = terrain_2d[:, ::-1]

    transformed_terrain = jnp.where(
        blue_perspective,
        mirrored_terrain,
        terrain_2d
    ).reshape(-1)

    return jnp.concatenate([
        transformed_terrain,
        unit_features
    ])

def make_symmetric_observation_batch(
    state,
    perspective_team
):
    return jax.vmap(
        symmetric_observation_one,
        in_axes=(0, None)
    )(state, perspective_team)

make_symmetric_observation_batch_jit = jax.jit(
    make_symmetric_observation_batch
)

# ============================================================
# POLICY NETWORK
# ============================================================

def init_policy(key):
    k1, k2, k3, k4 = random.split(
        key,
        4
    )

    params = {
        "W1": random.normal(
            k1,
            (OBS_SIZE, HIDDEN1)
        ) * jnp.sqrt(
            2.0 / OBS_SIZE
        ),
        "b1": jnp.zeros((HIDDEN1,)),

        "W2": random.normal(
            k2,
            (HIDDEN1, HIDDEN2)
        ) * jnp.sqrt(
            2.0 / HIDDEN1
        ),
        "b2": jnp.zeros((HIDDEN2,)),

        "Wa": random.normal(
            k3,
            (HIDDEN2, ACTION_SIZE)
        ) * 0.01,

        "ba": jnp.zeros((ACTION_SIZE,)),

        "Wv": random.normal(
            k4,
            (HIDDEN2, 1)
        ) * 0.01,

        "bv": jnp.zeros((1,)),
    }

    params["ba"] = params["ba"].at[
        1::3
    ].set(-0.3)

    return params

# ============================================================
# POLICY FORWARD
# ============================================================

def policy_forward(params, obs):
    h1 = jnp.tanh(
        jnp.matmul(
            obs,
            params["W1"]
        ) + params["b1"]
    )

    h2 = jnp.tanh(
        jnp.matmul(
            h1,
            params["W2"]
        ) + params["b2"]
    )

    action_output = (
        jnp.matmul(
            h2,
            params["Wa"]
        ) + params["ba"]
    )

    value = (
        jnp.matmul(
            h2,
            params["Wv"]
        ) + params["bv"]
    )[:, 0]

    return action_output, value

policy_forward_jit = jax.jit(
    policy_forward
)

# ============================================================
# ACTION INDICES
# ============================================================

ANGLE_MEAN_INDICES = jnp.arange(
    0,
    ACTION_SIZE,
    3
)

ANGLE_LOGSTD_INDICES = jnp.arange(
    1,
    ACTION_SIZE,
    3
)

ATTACK_INDICES = jnp.arange(
    2,
    ACTION_SIZE,
    3
)

# ============================================================
# LOCAL ACTION -> WORLD ACTION
# ============================================================

def local_to_world_action(
    action,
    perspective_team
):
    blue_perspective = (
        perspective_team == 1.0
    )

    local_dx = action[:, 0::3]
    local_dz = action[:, 1::3]
    attack = action[:, 2::3]

    world_dx = jnp.where(
        blue_perspective,
        -local_dx,
        local_dx
    )

    world_dz = local_dz

    world_action = jnp.zeros_like(action)

    world_action = world_action.at[
        :, 0::3
    ].set(world_dx)

    world_action = world_action.at[
        :, 1::3
    ].set(world_dz)

    world_action = world_action.at[
        :, 2::3
    ].set(attack)

    return world_action

# ============================================================
# ANGLE WRAP
# ============================================================

def wrap_angle(x):
    return (
        (x + jnp.pi)
        % (2.0 * jnp.pi)
        - jnp.pi
    )

# ============================================================
# ACTION LOG PROBABILITY
# ============================================================

def action_logprob(
    action_output,
    local_action
):
    angle_mean = action_output[
        :, ANGLE_MEAN_INDICES
    ]

    raw_logstd = action_output[
        :, ANGLE_LOGSTD_INDICES
    ]

    logstd = jnp.clip(
        raw_logstd,
        -3.0,
        0.5
    )

    std = jnp.exp(logstd)

    dx = local_action[:, 0::3]
    dz = local_action[:, 1::3]

    action_angle = jnp.arctan2(
        dz,
        dx
    )

    difference = wrap_angle(
        action_angle - angle_mean
    )

    angle_log_prob = (
        -0.5 * (difference / std) ** 2
        - logstd
        - 0.5 * jnp.log(2.0 * jnp.pi)
    )

    angle_log_prob = jnp.sum(
        angle_log_prob,
        axis=1
    )

    attack_logits = action_output[
        :, ATTACK_INDICES
    ]

    attack_action = local_action[
        :, ATTACK_INDICES
    ]

    log_p_attack = -jnp.logaddexp(
        0.0,
        -attack_logits
    )

    log_p_no_attack = -jnp.logaddexp(
        0.0,
        attack_logits
    )

    attack_log_prob = (
        attack_action * log_p_attack
        + (1.0 - attack_action)
        * log_p_no_attack
    )

    attack_log_prob = jnp.sum(
        attack_log_prob,
        axis=1
    )

    return (
        angle_log_prob
        + attack_log_prob
    )

# ============================================================
# POLICY ENTROPY
# 100兵士分の合計ではなく1兵士あたり平均
# ============================================================

def policy_entropy(action_output):
    logstd = jnp.clip(
        action_output[
            :, ANGLE_LOGSTD_INDICES
        ],
        -3.0,
        0.5
    )

    angle_entropy = (
        logstd
        + 0.5 * jnp.log(
            2.0 * jnp.pi * jnp.e
        )
    )

    angle_entropy = jnp.sum(
        angle_entropy,
        axis=1
    )

    attack_logits = action_output[
        :, ATTACK_INDICES
    ]

    p = jax.nn.sigmoid(
        attack_logits
    )

    attack_entropy = -(
        p * jnp.log(p + 1e-8)
        + (1.0 - p)
        * jnp.log(1.0 - p + 1e-8)
    )

    attack_entropy = jnp.sum(
        attack_entropy,
        axis=1
    )

    total_entropy = (
        angle_entropy
        + attack_entropy
    )

    return (
        total_entropy
        / N_SOLDIERS_PER_TEAM
    )

# ============================================================
# SAMPLE ACTION
# ============================================================

def sample_action(
    params,
    obs,
    key
):
    action_output, value = policy_forward(
        params,
        obs
    )

    batch_size = obs.shape[0]

    key_noise, key_attack = random.split(
        key
    )

    angle_mean = action_output[
        :, ANGLE_MEAN_INDICES
    ]

    logstd = jnp.clip(
        action_output[
            :, ANGLE_LOGSTD_INDICES
        ],
        -3.0,
        0.5
    )

    std = jnp.exp(logstd)

    noise = random.normal(
        key_noise,
        (
            batch_size,
            N_SOLDIERS_PER_TEAM
        )
    )

    angle = (
        angle_mean
        + std * noise
    )

    dx = jnp.cos(angle)
    dz = jnp.sin(angle)

    attack_logits = action_output[
        :, ATTACK_INDICES
    ]

    attack_probability = jax.nn.sigmoid(
        attack_logits
    )

    attack = random.bernoulli(
        key_attack,
        attack_probability
    ).astype(jnp.float32)

    local_action = jnp.zeros(
        (
            batch_size,
            ACTION_SIZE
        ),
        dtype=jnp.float32
    )

    local_action = local_action.at[
        :, 0::3
    ].set(dx)

    local_action = local_action.at[
        :, 1::3
    ].set(dz)

    local_action = local_action.at[
        :, 2::3
    ].set(attack)

    log_prob = action_logprob(
        action_output,
        local_action
    )

    return (
        local_action,
        log_prob,
        value
    )

sample_action_jit = jax.jit(
    sample_action
)

# ============================================================
# RESET FINISHED ENVIRONMENTS
# ============================================================

def reset_finished_envs(
    state,
    done,
    key
):
    keys = random.split(
        key,
        N_ENVS
    )

    reset_state = reset_parallel(
        keys
    )

    def merge(old, new):
        if old.ndim == 1:
            mask = done
        else:
            mask = done.reshape(
                (N_ENVS,)
                + (1,) * (old.ndim - 1)
            )

        return jnp.where(
            mask,
            new,
            old
        )

    return jax.tree_util.tree_map(
        merge,
        state,
        reset_state
    )

reset_finished_envs_jit = jax.jit(
    reset_finished_envs
)

# ============================================================
# GAE
# ============================================================

def compute_gae(
    rewards,
    values,
    dones,
    last_values
):
    def scan_fn(carry, inputs):
        previous_gae, next_value = carry

        reward_t, value_t, done_t = inputs

        mask = (
            1.0
            - done_t.astype(jnp.float32)
        )

        delta = (
            reward_t
            + GAMMA
            * next_value
            * mask
            - value_t
        )

        new_gae = (
            delta
            + GAMMA
            * GAE_LAMBDA
            * mask
            * previous_gae
        )

        return (
            (new_gae, value_t),
            new_gae
        )

    initial_carry = (
        jnp.zeros_like(last_values),
        last_values
    )

    _, advantages_reversed = lax.scan(
        scan_fn,
        initial_carry,
        (
            rewards[::-1],
            values[::-1],
            dones[::-1]
        )
    )

    advantages = advantages_reversed[::-1]
    returns = advantages + values

    return advantages, returns

compute_gae_jit = jax.jit(
    compute_gae
)

# ============================================================
# ADAM
# ============================================================

def adam_init(params):
    m = jax.tree_util.tree_map(
        jnp.zeros_like,
        params
    )

    v = jax.tree_util.tree_map(
        jnp.zeros_like,
        params
    )

    return {
        "m": m,
        "v": v,
        "t": jnp.array(
            0,
            dtype=jnp.int32
        )
    }

def adam_update(
    params,
    grads,
    opt_state
):
    beta1 = 0.9
    beta2 = 0.999
    eps = 1e-8

    t = opt_state["t"] + 1

    m = jax.tree_util.tree_map(
        lambda om, g:
            beta1 * om
            + (1.0 - beta1) * g,
        opt_state["m"],
        grads
    )

    v = jax.tree_util.tree_map(
        lambda ov, g:
            beta2 * ov
            + (1.0 - beta2) * g * g,
        opt_state["v"],
        grads
    )

    m_hat = jax.tree_util.tree_map(
        lambda x:
            x / (1.0 - beta1 ** t),
        m
    )

    v_hat = jax.tree_util.tree_map(
        lambda x:
            x / (1.0 - beta2 ** t),
        v
    )

    new_params = jax.tree_util.tree_map(
        lambda p, mh, vh:
            p
            - LEARNING_RATE * mh
            / (jnp.sqrt(vh) + eps),
        params,
        m_hat,
        v_hat
    )

    return new_params, {
        "m": m,
        "v": v,
        "t": t
    }

# ============================================================
# PPO LOSS
# ============================================================

def ppo_loss(
    params,
    obs,
    local_actions,
    old_log_probs,
    advantages,
    returns,
    entropy_coef
):
    action_output, values = policy_forward(
        params,
        obs
    )

    new_log_probs = action_logprob(
        action_output,
        local_actions
    )

    entropy = policy_entropy(
        action_output
    )

    ratio = jnp.exp(
        new_log_probs
        - old_log_probs
    )

    unclipped = (
        ratio
        * advantages
    )

    clipped = (
        jnp.clip(
            ratio,
            1.0 - PPO_CLIP,
            1.0 + PPO_CLIP
        )
        * advantages
    )

    policy_loss = -jnp.mean(
        jnp.minimum(
            unclipped,
            clipped
        )
    )

    value_loss = (
        0.5
        * jnp.mean(
            (returns - values) ** 2
        )
    )

    entropy_mean = jnp.mean(
        entropy
    )

    total_loss = (
        policy_loss
        + VALUE_COEF * value_loss
        - entropy_coef * entropy_mean
    )

    return total_loss, (
        policy_loss,
        value_loss,
        entropy_mean
    )

ppo_loss_grad = jax.jit(
    jax.value_and_grad(
        ppo_loss,
        has_aux=True
    )
)

@jax.jit
def ppo_update_minibatch(
    params,
    opt_state,
    obs,
    local_actions,
    old_log_probs,
    advantages,
    returns,
    entropy_coef
):
    (loss, metrics), grads = ppo_loss_grad(
        params,
        obs,
        local_actions,
        old_log_probs,
        advantages,
        returns,
        entropy_coef
    )

    params, opt_state = adam_update(
        params,
        grads,
        opt_state
    )

    return (
        params,
        opt_state,
        loss,
        metrics
    )

# ============================================================
# INITIALIZE POLICY
# ============================================================

master_key = random.key(12345)

master_key, init_key = random.split(
    master_key
)

params = init_policy(
    init_key
)

opt_state = adam_init(
    params
)

master_key, reset_key = random.split(
    master_key
)

state = reset(
    reset_key
)

print("Initial battle times:")
print(state["time"])

initial_min_time = float(
    jnp.min(state["time"])
)

initial_max_time = float(
    jnp.max(state["time"])
)

print(
    "Min time:",
    initial_min_time
)

print(
    "Max time:",
    initial_max_time
)

assert (
    initial_max_time == 0.0
), "Initial state was not reset to 0 seconds."

print(
    "All environments start at 0.0 sec."
)
print()

# ============================================================
# ROLLOUT COLLECTION
# ============================================================

def collect_rollout(
    params,
    state,
    key
):
    observations = []
    local_actions = []
    log_probs = []
    values = []
    rewards = []
    dones = []
    terminal_reason = []

    for step in range(ROLLOUT_STEPS):
        red_obs = (
            make_symmetric_observation_batch_jit(
                state,
                0.0
            )
        )

        blue_obs = (
            make_symmetric_observation_batch_jit(
                state,
                1.0
            )
        )

        key, red_key, blue_key = random.split(
            key,
            3
        )

        (
            red_local_action,
            red_logp,
            red_value
        ) = sample_action_jit(
            params,
            red_obs,
            red_key
        )

        (
            blue_local_action,
            blue_logp,
            blue_value
        ) = sample_action_jit(
            params,
            blue_obs,
            blue_key
        )

        red_world_action = local_to_world_action(
            red_local_action,
            0.0
        )

        blue_world_action = local_to_world_action(
            blue_local_action,
            1.0
        )

        (
            next_state,
            red_reward,
            blue_reward,
            done
        ) = step_parallel(
            state,
            red_world_action,
            blue_world_action
        )

        end_time = next_state["time"]

        timeout_now = (
            done
            & (end_time >= MAX_TIME)
        )

        commander_kill_now = (
            done
            & (~timeout_now)
        )

        red_win_now = (
            commander_kill_now
            & (red_reward > 0.999)
        )

        blue_win_now = (
            commander_kill_now
            & (blue_reward > 0.999)
        )

        reason = jnp.zeros(
            N_ENVS,
            dtype=jnp.int32
        )

        reason = jnp.where(
            red_win_now,
            1,
            reason
        )

        reason = jnp.where(
            blue_win_now,
            2,
            reason
        )

        reason = jnp.where(
            timeout_now,
            3,
            reason
        )

        terminal_reason.append(
            reason
        )

        batch_obs = jnp.concatenate([
            red_obs,
            blue_obs
        ], axis=0)

        batch_local_actions = jnp.concatenate([
            red_local_action,
            blue_local_action
        ], axis=0)

        batch_logp = jnp.concatenate([
            red_logp,
            blue_logp
        ], axis=0)

        batch_values = jnp.concatenate([
            red_value,
            blue_value
        ], axis=0)

        batch_rewards = jnp.concatenate([
            red_reward,
            blue_reward
        ], axis=0)

        batch_done = jnp.concatenate([
            done,
            done
        ], axis=0)

        observations.append(
            batch_obs
        )

        local_actions.append(
            batch_local_actions
        )

        log_probs.append(
            batch_logp
        )

        values.append(
            batch_values
        )

        rewards.append(
            batch_rewards
        )

        dones.append(
            batch_done
        )

        key, reset_key = random.split(
            key
        )

        state = reset_finished_envs_jit(
            next_state,
            done,
            reset_key
        )

    observations = jnp.stack(
        observations
    )

    local_actions = jnp.stack(
        local_actions
    )

    log_probs = jnp.stack(
        log_probs
    )

    values = jnp.stack(
        values
    )

    rewards = jnp.stack(
        rewards
    )

    dones = jnp.stack(
        dones
    )

    terminal_reason = jnp.stack(
        terminal_reason
    )

    red_obs = (
        make_symmetric_observation_batch_jit(
            state,
            0.0
        )
    )

    blue_obs = (
        make_symmetric_observation_batch_jit(
            state,
            1.0
        )
    )

    _, red_last_value = policy_forward_jit(
        params,
        red_obs
    )

    _, blue_last_value = policy_forward_jit(
        params,
        blue_obs
    )

    last_values = jnp.concatenate([
        red_last_value,
        blue_last_value
    ])

    advantages, returns = compute_gae_jit(
        rewards,
        values,
        dones,
        last_values
    )

    advantages = (
        advantages
        - jnp.mean(advantages)
    ) / (
        jnp.std(advantages)
        + 1e-8
    )

    flat_obs = observations.reshape(
        (-1, OBS_SIZE)
    )

    flat_local_actions = local_actions.reshape(
        (-1, ACTION_SIZE)
    )

    flat_log_probs = log_probs.reshape(
        (-1,)
    )

    flat_advantages = advantages.reshape(
        (-1,)
    )

    flat_returns = returns.reshape(
        (-1,)
    )

    # ========================================================
    # TERMINAL STATISTICS
    # ========================================================

    red_win_mask = (
        terminal_reason == 1
    )

    blue_win_mask = (
        terminal_reason == 2
    )

    timeout_mask = (
        terminal_reason == 3
    )

    terminal_mask = (
        terminal_reason != 0
    )

    terminal_count = jnp.sum(
        terminal_mask
    )

    red_wins = jnp.sum(
        red_win_mask
    )

    blue_wins = jnp.sum(
        blue_win_mask
    )

    timeout_count = jnp.sum(
        timeout_mask
    )

    red_rewards = rewards[
        :, :N_ENVS
    ]

    # --------------------------------------------------------
    # Timeout base reward
    # reward shapingを含まない本来の兵力差報酬
    # --------------------------------------------------------

    base_timeout_rewards = (
        timeout_mask.astype(jnp.float32)
        * 0.0
    )

    # terminal_reward = total reward - shaping
    # timeout時のみ取り出すため、
    # shapingを後から分離する。
    #
    # step_oneのshapingは reward_red - terminal_component_red
    # なので、timeout時のtotalからshapingを除けば
    # 本来のtimeout rewardになる。

    # terminal_reason=3 の時点では
    # total reward = timeout base reward + shaping
    # ここでは最終的な学習報酬から
    # shaping分を分離する必要がある。
    #
    # そのため rollout中のred rewardから直接
    # base timeout rewardを推定するのではなく、
    # timeout terminal rewardの集計を行う。

    # 現在の環境では timeout reward は
    # reward_red の terminal component なので、
    # shapingを分離した値を得るために
    # terminal rewardとの差分を記録する。

    red_terminal_rewards = jnp.where(
        terminal_mask,
        red_rewards,
        0.0
    )

    # Total terminal reward
    red_terminal_reward_sum = jnp.sum(
        red_terminal_rewards
    )

    red_terminal_mean = (
        red_terminal_reward_sum
        / jnp.maximum(
            terminal_count,
            1
        )
    )

    # --------------------------------------------------------
    # Timeout total reward
    # --------------------------------------------------------

    timeout_total_rewards = jnp.where(
        timeout_mask,
        red_rewards,
        0.0
    )

    timeout_total_reward_sum = jnp.sum(
        timeout_total_rewards
    )

    timeout_total_reward_mean = (
        timeout_total_reward_sum
        / jnp.maximum(
            timeout_count,
            1
        )
    )

    # --------------------------------------------------------
    # シェーピング報酬の分離は、
    # 実際のrewardとterminal rewardを比較するだけでは
    # terminal step以外にも存在するため、
    # 現時点ではtimeoutのtotal rewardを表示する。
    #
    # 「TimeoutR」が以前より正確に
    # 「PPOに実際に渡されたtimeout時報酬」
    # を意味するようにする。
    # --------------------------------------------------------

    rollout_simulated_time = (
        ROLLOUT_STEPS * DT
    )

    return (
        state,
        key,
        flat_obs,
        flat_local_actions,
        flat_log_probs,
        flat_advantages,
        flat_returns,
        terminal_count,
        red_wins,
        blue_wins,
        timeout_count,
        red_terminal_mean,
        timeout_total_reward_mean,
        rollout_simulated_time
    )

# ============================================================
# TRAINING
# ============================================================

N_UPDATES = 30

cumulative_red_wins = 0
cumulative_blue_wins = 0
cumulative_timeouts = 0

print()
print("Starting PPO training...")
print()

for update in range(
    1,
    N_UPDATES + 1
):
    update_start = time.time()

    entropy_coef = get_entropy_coef(
        update
    )

    (
        state,
        master_key,
        obs_batch,
        local_action_batch,
        old_logprob_batch,
        advantage_batch,
        return_batch,
        terminal_count,
        red_wins,
        blue_wins,
        timeout_count,
        red_terminal_mean,
        timeout_total_reward_mean,
        rollout_simulated_time
    ) = collect_rollout(
        params,
        state,
        master_key
    )

    batch_size = obs_batch.shape[0]
    epoch_metrics = []

    for epoch in range(
        PPO_EPOCHS
    ):
        master_key, shuffle_key = random.split(
            master_key,
            2
        )

        permutation = random.permutation(
            shuffle_key,
            batch_size
        )

        for minibatch in range(
            MINIBATCHES
        ):
            start_index = (
                minibatch
                * MINIBATCH_SIZE
            )

            end_index = (
                start_index
                + MINIBATCH_SIZE
            )

            indices = permutation[
                start_index:end_index
            ]

            mb_obs = obs_batch[
                indices
            ]

            mb_local_actions = local_action_batch[
                indices
            ]

            mb_old_logprob = old_logprob_batch[
                indices
            ]

            mb_advantage = advantage_batch[
                indices
            ]

            mb_returns = return_batch[
                indices
            ]

            (
                params,
                opt_state,
                loss,
                metrics
            ) = ppo_update_minibatch(
                params,
                opt_state,
                mb_obs,
                mb_local_actions,
                mb_old_logprob,
                mb_advantage,
                mb_returns,
                entropy_coef
            )

            epoch_metrics.append(
                metrics
            )

    metrics_array = jnp.stack([
        jnp.asarray(m)
        for m in epoch_metrics
    ])

    mean_policy_loss = jnp.mean(
        metrics_array[:, 0]
    )

    mean_value_loss = jnp.mean(
        metrics_array[:, 1]
    )

    mean_entropy = jnp.mean(
        metrics_array[:, 2]
    )

    terminal_count_i = int(
        terminal_count
    )

    red_wins_i = int(
        red_wins
    )

    blue_wins_i = int(
        blue_wins
    )

    timeout_count_i = int(
        timeout_count
    )

    cumulative_red_wins += (
        red_wins_i
    )

    cumulative_blue_wins += (
        blue_wins_i
    )

    cumulative_timeouts += (
        timeout_count_i
    )

    if terminal_count_i > 0:
        red_win_rate = (
            red_wins_i
            / terminal_count_i
        )

        blue_win_rate = (
            blue_wins_i
            / terminal_count_i
        )

        timeout_rate = (
            timeout_count_i
            / terminal_count_i
        )
    else:
        red_win_rate = 0.0
        blue_win_rate = 0.0
        timeout_rate = 0.0

    cumulative_games = (
        cumulative_red_wins
        + cumulative_blue_wins
        + cumulative_timeouts
    )

    if cumulative_games > 0:
        cumulative_red_win_rate = (
            cumulative_red_wins
            / cumulative_games
        )

        cumulative_blue_win_rate = (
            cumulative_blue_wins
            / cumulative_games
        )

        cumulative_timeout_rate = (
            cumulative_timeouts
            / cumulative_games
        )
    else:
        cumulative_red_win_rate = 0.0
        cumulative_blue_win_rate = 0.0
        cumulative_timeout_rate = 0.0

    elapsed = (
        time.time()
        - update_start
    )

    print(
        f"Update {update:3d} | "
        f"time {elapsed:6.2f}s | "
        f"sim {rollout_simulated_time:5.1f}s | "
        f"terminal {terminal_count_i:3d} | "
        f"RedWin {red_win_rate:6.2%} | "
        f"BlueWin {blue_win_rate:6.2%} | "
        f"Timeout {timeout_rate:6.2%} | "
        f"RedTermR {float(red_terminal_mean): .4f} | "
        f"TimeoutR {float(timeout_total_reward_mean): .4f} | "
        f"CumR {cumulative_red_win_rate:6.2%} | "
        f"CumB {cumulative_blue_win_rate:6.2%} | "
        f"CumT {cumulative_timeout_rate:6.2%} | "
        f"policy {float(mean_policy_loss): .5f} | "
        f"value {float(mean_value_loss): .5f} | "
        f"entropy {float(mean_entropy): .5f} | "
        f"ent_coef {entropy_coef:.4f}"
    )


PPO SELF-PLAY
Observation : 2120
Action      : 300
Environments: 32
Rollout     : 900
PPO batch   : 57600
Minibatch   : 7200
Gamma       : 0.999
GAE lambda  : 0.97
Rollout time: 180.0 sec
Max battle : 180.0 sec
Shaping coef: 0.005
Entropy coef: 0.02 -> 0.002

REWARD VERIFICATION
Commander kill (Red): 1.0
Commander kill (Blue): -1.0
Timeout 72 vs 54: 0.18
Timeout 54 vs 72: -0.18
Timeout 50 vs 50: 0.0
Ongoing battle: 0.0

ALL REWARD TESTS PASSED.

Initial battle times:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0.]
Min time: 0.0
Max time: 0.0
All environments start at 0.0 sec.


Starting PPO training...

Update   1 | time  50.85s | sim 180.0s | terminal   0 | RedWin  0.00% | BlueWin  0.00% | Timeout  0.00% | RedTermR  0.0000 | TimeoutR  0.0000 | CumR  0.00% | CumB  0.00% | CumT  0.00% | policy  0.03345 | value  0.00027 | entropy  1.82170 | ent_coef 0.0199
Update   2 | time  21.76s | sim 180.0s | terminal  32 | RedWin  0.00% | BlueWin  